# Data Visualization Workflow
Use this notebook to load an already processed dataset, optionally reload a range file, and open the interactive visualization tools.


In [ ]:
# Activate intractive functionality of matplotlib
%matplotlib ipympl
# Activate auto reload 
%load_ext autoreload
%autoreload 2
%reload_ext autoreload
# import libraries
import os
import numpy as np
import subprocess
from ipywidgets import widgets
from IPython.display import display
from ipywidgets import fixed, interact_manual
import warnings
# Ignore all warnings
warnings.filterwarnings("ignore")

# Local module and scripts
from pyccapt.calibration.core import widgets as wd
from pyccapt.calibration.data_tools import data_tools
from pyccapt.calibration.tutorials.tutorials_helpers import helper_data_loader
from pyccapt.calibration.tutorials.tutorials_helpers import helper_visualization
from pyccapt.calibration.core import share_variables
from pyccapt.calibration.core import ion_selection

In [ ]:
from pyccapt.calibration.core import plot_style
plot_style.set_paper_style('paper')    # round-number publication axes
# or
# plot_style.set_paper_style('normal')   # plain matplotlib defaults

If HDF5 loading fails because `pytables` is missing, install it in a separate cell before continuing:

`!conda install --yes --prefix {sys.prefix} pytables`


Create the shared `variables` object first. The loaded dataset and range table are stored here for the visualization helpers.


In [ ]:
# Create the shared state container used by the visualization helpers.
variables = share_variables.Variables()

Choose the dataset you want to visualize. This notebook works best with data that has already been cropped and calibrated.


In [ ]:
button = widgets.Button(description='Load dataset')

@button.on_click
def open_file_on_click_r(b):
    global dataset_path
    folder_path = variables.last_directory
    script = '..//..//data_tools//run_dataset_path_qt.py'
    cmd = f"python {script} {folder_path}"
    result = subprocess.run(cmd, capture_output=True, text=True, shell=True)
    dataset_path = result.stdout.strip()
    variables.last_directory = dataset_path

button

If you already have a saved range file (.h5, .rrng, or .rng), load it here so the ion labels and colors are restored before visualization.

In [ ]:
button_r = widgets.Button(description='Load range dataset')

@button_r.on_click
def open_file_on_click_r(b):
    global range_path
    folder_path = variables.last_directory
    script = '..//..//data_tools//run_dataset_path_qt.py'
    result = subprocess.run(
        ['python', script, folder_path, 'range'],
        capture_output=True,
        text=True,
        shell=False,
    )
    selected_path = result.stdout.strip()
    if selected_path and selected_path != 'No file chosen':
        # If the user picked a MATLAB Atom-Probe-Toolbox figure (.fig),
        # convert it on-the-fly into a PyCCAPT range .h5 and use that.
        if selected_path.lower().endswith('.fig'):
            from pathlib import Path
            from pyccapt.calibration.leap_tools import matlab_fig_range
            fig_path = Path(selected_path)
            out_h5 = fig_path.with_name(f"{fig_path.stem}_range.h5")
            frame = matlab_fig_range.fig_to_range_dataframe(fig_path)
            matlab_fig_range.write_pyccapt_range(frame, out_h5, also_csv=True)
            print(f"Converted MATLAB .fig to PyCCAPT range: {out_h5} ({len(frame)} ranges)")
            selected_path = str(out_h5)
        range_path = selected_path
        variables.last_directory = range_path

button_r

## Load Data And Optional Range Files
Set the instrument metadata, load the dataset, and optionally reload a previously saved range table.


Review the dataset settings before loading. The selected values control the derived arrays used by the visualization helpers.


In [ ]:
# create an object for selection of instrument specifications of the dataset
tdc, pulse_mode, flight_path_length, t0, max_mc, det_diam = wd.dataset_instrument_specification_selection()
# Toggle for loading the raw /tdc group alongside /dld so raw rows can be
# saved together with the calibrated dld dataset.
load_tdc_raw = wd.load_tdc_raw_selection()

# Display lists and comboboxes to selected instrument specifications
display(tdc, pulse_mode, flight_path_length, t0, max_mc, load_tdc_raw)

In [ ]:
# Load the dataset with the selected settings and preview the dataset and range table.
helper_data_loader.load_data(dataset_path, max_mc.value, flight_path_length.value, pulse_mode.value, tdc.value, variables,
                             processing_mode=False, load_tdc_raw=load_tdc_raw.value)
data_tools.extract_data(variables.data, variables, flight_path_length.value, max_mc.value)
# If a range file was chosen, load it into the shared state and preview it.
if 'range_path' in globals():
    variables.range_data = data_tools.read_range(range_path)
display(variables.data)
display(variables.range_data)
if variables.data_tdc is not None:
    print(f"Linked raw tdc rows in memory: {len(variables.data_tdc)}")

In [ ]:
# Preview the range table with colors applied before saving or visualization.
display(variables.range_data.style.map(ion_selection.display_color, subset=['color']))

Save the current range table from the widget below once the ion list looks correct.


In [ ]:
# Save the current range table interactively.
interact_manual_range = interact_manual.options(manual_name="Save range")
_ = interact_manual_range(data_tools.save_range, variables=fixed(variables));


In [ ]:
variables.data

Use the next widget to export the current dataset in any additional formats you need.


In [ ]:
# Save the dataset interactively in the output formats you want.
# The default export range is the full dataset; edit the indices below to save a smaller subset.
# When raw tdc was loaded, set save_tdc=True to also write linked raw rows under /tdc.
_export_last_index = max(0, len(variables.data) - 1)
interact_manual_data = interact_manual.options(manual_name="Save data")
interact_manual_data(data_tools.save_data, data=fixed(variables.data), variables=fixed(variables),
                name=widgets.Text(value=variables.result_data_name),
                hdf=widgets.Dropdown(options=[('True', True), ('False', False)]),
                epos=widgets.Dropdown(options=[('False', False), ('True', True)]),
                pos=widgets.Dropdown(options=[('False', False), ('True', True)]),
                ato_6v=widgets.Dropdown(options=[('False', False), ('True', True)]),
                csv=widgets.Dropdown(options=[('False', False), ('True', True)]),
                save_tdc=widgets.Dropdown(options=[('False', False), ('True', True)], value=(variables.data_tdc is not None), description='Save raw tdc:'),
                start_index=widgets.BoundedIntText(value=0, min=0, max=_export_last_index, description='From index'),
                end_index=widgets.BoundedIntText(value=_export_last_index, min=0, max=_export_last_index, description='To index'),
               temp=fixed(False));


## Visualization
The final cells refresh the extracted arrays and open the interactive visualization interface.

The `3D` and `Iso surface` panels now include optional Min-Max clustering controls. Turn clustering on, enter the ion or element labels for the precipitate of interest, and PyCCAPT will split that selected population into two precipitate segments in the reconstruction view.

In [ ]:
# Refresh the extracted arrays before launching the visualization helper.
data_tools.extract_data(variables.data, variables, flight_path_length.value, max_mc.value)

In [ ]:
# Open the main visualization interface.
helper_visualization.call_visualization(variables)

In [ ]:
from pyccapt.calibration.core import mc_plot
hist = variables.data['mc (Da)'].to_numpy()
mc_hist = mc_plot.AptHistPlotter(hist[hist < 40], variables)
mc_hist.plot_histogram(bin_width=0.01, label='mc', steps='stepfilled', log=True)
mc_hist.find_peaks_and_widths()
mc_hist.plot_hist_info_legend(loc='right')
mc_hist.plot_line_hist()
mc_hist.manual_background_fit()

In [ ]:
# Display the plotly reconstruction object so you can inspect or embed it.
variables.plotly_3d_reconstruction

In [ ]:
from IPython.display import display, HTML
display(HTML(variables.animation_detector_html))